In [38]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from sklearn.metrics import classification_report,accuracy_score
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns
from sklearn.model_selection import GridSearchCV
import joblib
from sklearn.ensemble import RandomForestClassifier
from pickle import dump
from sklearn.linear_model import LogisticRegression





In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv')
pd.set_option('display.max_columns',None)
df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [3]:
def apply_preprocess(df):
    df = df.drop("package_name", axis=1)
    df["review"] = df["review"].str.strip().str.lower()

    return df

In [4]:
total_data = apply_preprocess(df)

In [5]:
total_data.head()

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0


In [6]:
X = total_data['review']
y = total_data['polarity']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [8]:
vectorizer = CountVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

In [9]:
X_train_vec

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(712, 3310))

Se decide empezar utilizando la implementación multinomial del modelo, ya que su uso se adapta a datos discretos que representan recuentos o proporciones. 

In [22]:
bayes = MultinomialNB().fit(X_train_vec, y_train)
y_pred = bayes.predict(X_test_vec)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test, y_pred))


0.8156424581005587
              precision    recall  f1-score   support

           0       0.84      0.90      0.87       126
           1       0.73      0.60      0.66        53

    accuracy                           0.82       179
   macro avg       0.79      0.75      0.77       179
weighted avg       0.81      0.82      0.81       179



In [23]:
nb_ber = BernoulliNB().fit(X_train_vec,y_train)
y_pred_ber = nb_ber.predict(X_test_vec)
print(accuracy_score(y_test,y_pred_ber))
print(classification_report(y_test,y_pred_ber))

0.770949720670391
              precision    recall  f1-score   support

           0       0.79      0.93      0.85       126
           1       0.70      0.40      0.51        53

    accuracy                           0.77       179
   macro avg       0.74      0.66      0.68       179
weighted avg       0.76      0.77      0.75       179



In [24]:
nb_gauss = GaussianNB().fit(X_train_vec, y_train)
y_pred_gauss = nb_gauss.predict(X_test_vec)
print(accuracy_score(y_test,y_pred_gauss))
print(classification_report(y_test, y_pred_gauss))

0.8044692737430168
              precision    recall  f1-score   support

           0       0.85      0.88      0.86       126
           1       0.69      0.62      0.65        53

    accuracy                           0.80       179
   macro avg       0.77      0.75      0.76       179
weighted avg       0.80      0.80      0.80       179



Aunque la implementación GaussianNB tiene mejor accuracy score y además, un f1 score bastante cercano al de Multinomial, y superior al de Bernoulli, en este proyecto, elegiré Multinomial NB ya que tiene una mejor justificación teórica. 

Optimizar modelo con Random Forest

In [34]:
randomf = RandomForestClassifier(n_estimators=50,random_state=42,n_jobs=-1)
randomf.fit(X_train_vec,y_train)
y_pred_rf = randomf.predict(X_test_vec)
print(accuracy_score(y_test,y_pred_rf))
print(classification_report(y_test,y_pred_rf))

0.8212290502793296
              precision    recall  f1-score   support

           0       0.91      0.83      0.87       126
           1       0.67      0.79      0.72        53

    accuracy                           0.82       179
   macro avg       0.79      0.81      0.80       179
weighted avg       0.83      0.82      0.83       179



RandomForest logro optimizar mejor los resultados. Fue mejor capturando reseñas positivas que MultinomialNB

Exp. Modelo

In [36]:
best_model = randomf

In [37]:
joblib.dump(randomf, 'modelo_clasificacion_sentimientos.pkl' )

['modelo_clasificacion_sentimientos.pkl']

Explorar otras opciones; se utilizará LogisticRegression ya que también permite clasificar texto y maneja bien la alta dimensionalidad. 

In [41]:
log_reg = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
log_reg.fit(X_train_vec,y_train)
y_pred_log = log_reg.predict(X_test_vec)
print(accuracy_score(y_test,y_pred_log))
print(classification_report(y_test,y_pred_log))

0.8268156424581006
              precision    recall  f1-score   support

           0       0.91      0.84      0.87       126
           1       0.68      0.79      0.73        53

    accuracy                           0.83       179
   macro avg       0.79      0.82      0.80       179
weighted avg       0.84      0.83      0.83       179



Tiene un resultado similar al que obtuvimos con RF pero mejora un poco mas el performance capturando reseñas positivas. 